# Build Master Clips CSV

Produces a single CSV with one row per BST-cleaned ShuttleSet clip, carrying every field needed to derive any (split, taxonomy) combination at collation time:

- `clip_stem` (e.g. `1_1_30_2`) -- the unique on-disk identifier
- `match`, `set_id`, `rally`, `ball_round`, `vid` -- ShuttleSet annotation keys
- `raw_type_en` -- the 19 raw stroke types (English)
- `player_side` -- `Top` / `Bottom`, derived via `pipeline.player_mapping.collect_shots` (matches production exactly, including the set-3 11-point court switch)
- `split_bst_baseline` -- from `pipeline.config.SPLITS` (BST's by-video-id split, currently encoded on disk under `dataset_npy_between_2_hits_with_max_limits/{train,val,test}/...`)
- `split_v2` -- from `notebooks/shuttleset_splits_v2.csv` (Isiah's player-overlap-minimised split)
- `aroundhead`, `backhand` -- from `shuttleset_splits_v2.csv` (already cleaned/binarised)

This is the source of truth for the verification script and for the refactored `collate_npy()`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add bst_x to path so we can import pipeline modules
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src' / 'bst_x'))

from pipeline.config import (
    SPLITS,             # {split_name: [video_ids]} after EXCLUDED_VIDEOS strip
    EXCLUDED_VIDEOS,    # set[int] of fully-dropped match IDs
    REMOVED_SHOTS,      # set[(vid, set_num, rally, ball_round)] of dropped shots
    STROKE_TYPES_19_ZH, # all 19 raw stroke types in Chinese (matches CSV `type` col)
    SET_INFO_DIR,       # ShuttleSet/set/ -- contains match.csv + per-match folders
)
from pipeline.player_mapping import collect_shots

print(f'EXCLUDED_VIDEOS: {sorted(EXCLUDED_VIDEOS)}')
print(f'REMOVED_SHOTS:   {len(REMOVED_SHOTS)} entries')
print(f'SPLITS:          train={len(SPLITS["train"])}, val={len(SPLITS["val"])}, test={len(SPLITS["test"])} videos')
print(f'SET_INFO_DIR:    {SET_INFO_DIR}')

EXCLUDED_VIDEOS: [9, 10, 12, 27]
REMOVED_SHOTS:   5 entries
SPLITS:          train=30, val=5, test=5 videos
SET_INFO_DIR:    /home/ariel/Documents/COSC594/badminton_stroke_classification/src/bst_x/ShuttleSet/set


## 1. Load match metadata
Pull `(id, video, downcourt)` from `match.csv`. `downcourt` (bool) tells us whether player A starts on top -- needed by `collect_shots` for the Top/Bottom mapping.

In [2]:
match_df = pd.read_csv(SET_INFO_DIR / 'match.csv')[['id', 'video', 'downcourt']]
match_df['downcourt'] = match_df['downcourt'].astype(bool)
match_df = match_df.set_index('id')

# vid -> split_bst_baseline lookup
vid_to_split = {}
for split_name, vids in SPLITS.items():
    for v in vids:
        vid_to_split[v] = split_name

print(f'Loaded {len(match_df)} matches')
print(f'Excluded video IDs absent from SPLITS as expected: {sorted(set(match_df.index) - set(vid_to_split.keys()))}')

Loaded 44 matches
Excluded video IDs absent from SPLITS as expected: [9, 10, 12, 27]


## 2. Collect shots per video via the production `collect_shots`
Reusing `pipeline.player_mapping.collect_shots` guarantees the `player_side` (Top/Bottom) values are identical to whatever was used when the existing on-disk clips were placed into folders. If we re-implemented the mapping here we'd risk subtle disagreements that the verification script would later flag.

`collect_shots` returns columns: `set, rally, ball_round, frame_num, roundscore_A, roundscore_B, player ('Top'/'Bottom'), type (English)`.

In [3]:
rows = []
for vid, v_info in match_df.iterrows():
    if vid in EXCLUDED_VIDEOS:
        continue
    shots = collect_shots(SET_INFO_DIR, v_info, STROKE_TYPES_19_ZH)
    if shots.empty:
        print(f'  WARNING: vid={vid} ({v_info["video"]}) returned no shots')
        continue
    shots = shots.copy()
    shots['vid'] = vid
    shots['match'] = v_info['video']
    rows.append(shots)

raw_df = pd.concat(rows, ignore_index=True)
print(f'\nCollected {len(raw_df):,} shots across {raw_df["vid"].nunique()} videos')
print(f'Player split: {raw_df["player"].value_counts().to_dict()}')


Collected 33,486 shots across 40 videos
Player split: {'Bottom': 16749, 'Top': 16737}


## 3. Apply REMOVED_SHOTS
These are 5 individually flagged bad shots from `flaw_shot_records.csv` -- already pulled into `pipeline.config.REMOVED_SHOTS`. Each tuple is `(vid, set_num, rally, ball_round)`.

In [4]:
before = len(raw_df)
removed_keys = set(REMOVED_SHOTS)
shot_keys = list(zip(
    raw_df['vid'].astype(int),
    raw_df['set'].astype(int),
    raw_df['rally'].astype(int),
    raw_df['ball_round'].astype(int),
))
keep_mask = [k not in removed_keys for k in shot_keys]
raw_df = raw_df[keep_mask].reset_index(drop=True)
after = len(raw_df)
print(f'Removed {before - after} shots via REMOVED_SHOTS  ({before:,} -> {after:,})')

Removed 5 shots via REMOVED_SHOTS  (33,486 -> 33,481)


## 4. Build `clip_stem` and add the BST baseline split
Clip stems are `{vid}_{set_num}_{rally}_{int(ball_round)}` -- exactly the naming convention used by clip extraction (`prepare_train_on_shuttleset.py` line 782 derives `vid = int(clip_stem.split("_", 1)[0])`).

In [5]:
raw_df['ball_round'] = raw_df['ball_round'].astype(int)
raw_df['clip_stem'] = (
    raw_df['vid'].astype(str) + '_'
    + raw_df['set'].astype(str) + '_'
    + raw_df['rally'].astype(str) + '_'
    + raw_df['ball_round'].astype(str)
)
raw_df['split_bst_baseline'] = raw_df['vid'].map(vid_to_split)
raw_df['set_id'] = 'set' + raw_df['set'].astype(str)

raw_df = raw_df.rename(columns={'player': 'player_side', 'type': 'raw_type_en'})

# Sanity: every row should have a baseline split (no NaNs)
missing_split = raw_df['split_bst_baseline'].isna().sum()
assert missing_split == 0, f'{missing_split} rows have no split_bst_baseline'

# Sanity: clip stems should be unique
dup_stems = raw_df['clip_stem'].duplicated().sum()
assert dup_stems == 0, f'{dup_stems} duplicate clip stems'

print(f'Built {len(raw_df):,} unique clip rows')
print(f'\nsplit_bst_baseline counts:')
print(raw_df['split_bst_baseline'].value_counts())
print(f'\nraw_type_en counts (top 10):')
print(raw_df['raw_type_en'].value_counts().head(10))

Built 33,481 unique clip rows

split_bst_baseline counts:
split_bst_baseline
train    25741
val       4241
test      3499
Name: count, dtype: int64

raw_type_en counts (top 10):
raw_type_en
net_shot         5824
lob              4879
return_net       3374
clear            2661
push             2652
smash            2362
drop             1979
short_service    1858
wrist_smash      1559
unknown          1278
Name: count, dtype: int64


## 5. Join Isiah's v2 split + binarised flags
`shuttleset_splits_v2.csv` has 32,203 BST-cleaned non-Unknown clips with `split` (re-stratified for player overlap) plus `aroundhead`/`backhand` already binarised.

Left-join on `(match, set_id, rally, ball_round)`. Unknown clips will have `split_v2` NaN -- expected; `collate_npy` filters them out when `drop_unknown=True`.

(v1 and v2 were patched in place on 2026-04-20 to fix 3 stale match-name variants from an older filesystem state and drop 1 true duplicate row; the join now needs no live fix-ups. See notebook 01 markdown for the post-export patch summary.)

In [6]:
v2 = pd.read_csv(REPO_ROOT / 'notebooks' / 'shuttleset_splits_v2.csv')
v2['ball_round'] = v2['ball_round'].astype(int)
v2 = v2.rename(columns={'split': 'split_v2'})

# Sanity: v2 should be clean post-patch (no stale names, no duplicates).
v2_keys = ['match', 'set_id', 'rally', 'ball_round']
assert v2.duplicated(subset=v2_keys).sum() == 0, 'v2 has duplicate keys'

v2_cols = v2_keys + ['split_v2', 'aroundhead', 'backhand']
master = raw_df.merge(v2[v2_cols], on=v2_keys, how='left')

n_no_v2 = master['split_v2'].isna().sum()
n_unknown = (master['raw_type_en'] == 'unknown').sum()
print(f'Rows missing split_v2: {n_no_v2:,}')
print(f'Rows with raw_type_en == "unknown": {n_unknown:,}')
if n_no_v2 != n_unknown:
    diff = master[master['split_v2'].isna() & (master['raw_type_en'] != 'unknown')]
    print(f'\nWARNING: {len(diff)} non-unknown rows lack split_v2 (expected 0):')
    print(diff[['vid', 'match', 'set_id', 'rally', 'ball_round', 'raw_type_en']].head(20))

Rows missing split_v2: 1,278
Rows with raw_type_en == "unknown": 1,278


## 6. Reorder columns and save

In [7]:
out_cols = [
    'clip_stem',
    'vid', 'match', 'set_id', 'rally', 'ball_round',
    'raw_type_en', 'player_side',
    'split_bst_baseline', 'split_v2',
    'aroundhead', 'backhand',
]
out = master[out_cols].copy()
out_path = REPO_ROOT / 'notebooks' / 'clips_master.csv'
out.to_csv(out_path, index=False)
print(f'Saved {len(out):,} rows to {out_path}')
print(f'\nColumns: {list(out.columns)}')
print(f'\nSample rows:')
out.head()

Saved 33,481 rows to /home/ariel/Documents/COSC594/badminton_stroke_classification/notebooks/clips_master.csv

Columns: ['clip_stem', 'vid', 'match', 'set_id', 'rally', 'ball_round', 'raw_type_en', 'player_side', 'split_bst_baseline', 'split_v2', 'aroundhead', 'backhand']

Sample rows:


,clip_stem,vid,match,set_id,rally,ball_round,raw_type_en,player_side,split_bst_baseline,split_v2,aroundhead,backhand
0,1_1_1_1,1,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_F...,set1,1,1,short_service,Bottom,train,train,0.0,1.0
1,1_1_1_2,1,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_F...,set1,1,2,clear,Top,train,train,0.0,1.0
2,1_1_2_1,1,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_F...,set1,2,1,short_service,Bottom,train,train,0.0,1.0
3,1_1_2_2,1,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_F...,set1,2,2,push,Top,train,train,0.0,1.0
4,1_1_2_3,1,Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_F...,set1,2,3,smash,Bottom,train,train,0.0,0.0


## 7. Cross-check vs. notebook 01 / 02 totals

Expected counts after BST cleaning:
- 33,483 total clips (after dropping 4 excluded matches and 5 named shots, before unknown drop)
- 32,204 non-unknown (matches `shuttleset_splits.csv` and `shuttleset_splits_v2.csv` row counts)

In [8]:
n_total = len(out)
n_non_unknown = (out['raw_type_en'] != 'unknown').sum()
n_v2_assigned = out['split_v2'].notna().sum()

# Notebook 01 reported 33,483 clips after BST cleaning at the time it ran.
# The raw ShuttleSet CSVs have drifted by ~2 shots since then (annotators
# fix a typo here and there); we tolerate small drift but loud-warn on a
# big one so it doesn't slip silently.
EXPECTED_TOTAL = 33_483
EXPECTED_NON_UNKNOWN = 32_203  # = patched v2 row count (post dup drop)

drift = abs(n_total - EXPECTED_TOTAL)
print(f'Total rows:              {n_total:,}  (notebook 01 baseline 33,483, drift {drift})')
print(f'Non-unknown rows:        {n_non_unknown:,}  (target 32,203)')
print(f'Rows with v2 assignment: {n_v2_assigned:,}  (should equal patched v2 row count = 32,203)')

assert drift <= 5, f'Total {n_total} drifts >5 from 33,483 -- investigate raw CSVs'
assert n_non_unknown == n_v2_assigned, (
    f'Non-unknown count ({n_non_unknown}) != v2-assigned count ({n_v2_assigned}); '
    'some non-unknown clips failed to join to v2'
)
print('\nMaster CSV ready for verification + collation.')

Total rows:              33,481  (notebook 01 baseline 33,483, drift 2)
Non-unknown rows:        32,203  (target 32,203)
Rows with v2 assignment: 32,203  (should equal patched v2 row count = 32,203)

Master CSV ready for verification + collation.
